# 4. Machine Learning con GridSearch
Probamos varios modelos y sus hiperparámetros para encontrar el mejor.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB

In [ ]:
df = pd.read_csv("data/titanic_fe.csv")

X = df.drop("Survived", axis=1)
y = df["Survived"]

# test_size=0.2 -> 20% para prueba; random_state fija la división para poder repetir resultados
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape, X_test.shape

## Diccionario de modelos e hiperparámetros

In [ ]:
modelos = {
    'Logistic Regression': {
        'model': LogisticRegression(),
        'params': {'model__C': [0.1], 'model__max_iter': [1000]}
    },
    'Support Vector Classifier': {
        'model': SVC(),
        'params': {'model__kernel': ['linear', 'poly', 'rbf', 'sigmoid'], 'model__C': [0.1, 1, 10]}
    },
    'Decision Tree Classifier': {
        'model': DecisionTreeClassifier(),
        'params': {'model__splitter': ['best', 'random'], 'model__max_depth': [None, 1, 2, 3, 4]}
    },
    'Random Forest Classifier': {
        'model': RandomForestClassifier(),
        'params': {
            'model__n_estimators': [10, 100],
            'model__max_depth': [None, 1, 2, 3, 4],
            'model__max_features': ['sqrt', 'log2']  # 'auto' ya no existe en scikit-learn reciente
        }
    },
    'Gradient Boosting Classifier': {
        'model': GradientBoostingClassifier(),
        'params': {'model__n_estimators': [10, 100], 'model__max_depth': [None, 1, 2, 3, 4]}
    },
    'AdaBoost Classifier': {
        'model': AdaBoostClassifier(),
        'params': {'model__n_estimators': [10, 100]}
    },
    'K-Nearest Neighbors Classifier': {
        'model': KNeighborsClassifier(),
        'params': {'model__n_neighbors': [3, 5, 7]}
    },
    'GaussianNB': {
        'model': GaussianNB(),
        'params': {}
    },
    'Naive Bayes Classifier': {
        'model': BernoulliNB(),
        'params': {'model__alpha': [0.1, 1.0, 10.0]}
    }
}

# XGBoost y LightGBM se agregan si están instalados (pip install xgboost lightgbm)
try:
    from xgboost import XGBClassifier
    modelos['XGBoost Classifier'] = {
        'model': XGBClassifier(),
        'params': {'model__n_estimators': [10, 100], 'model__max_depth': [None, 1, 2, 3]}
    }
except ImportError:
    print("xgboost no está instalado, se omite")

try:
    from lightgbm import LGBMClassifier
    modelos['LGBM Classifier'] = {
        'model': LGBMClassifier(),
        'params': {
            'model__n_estimators': [10, 100],
            'model__max_depth': [None, 1, 2, 3],
            'model__learning_rate': [0.1, 0.2, 0.3],
            'model__verbose': [-1]
        }
    }
except ImportError:
    print("lightgbm no está instalado, se omite")

## GridSearch
Cada modelo va dentro de un `Pipeline` con escalado. `cv=5` hace validación cruzada en 5 partes, `n_jobs=-1` usa todos los núcleos y `verbose=0` no imprime detalles.

In [ ]:
resultados = []
mejores_modelos = {}

for nombre, config in modelos.items():
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('model', config['model'])
    ])
    grid = GridSearchCV(pipe, config['params'], cv=5, scoring='accuracy', n_jobs=-1, verbose=0)
    grid.fit(X_train, y_train)

    mejores_modelos[nombre] = grid.best_estimator_
    resultados.append({
        'modelo': nombre,
        'accuracy_cv': round(grid.best_score_, 4),
        'accuracy_test': round(accuracy_score(y_test, grid.predict(X_test)), 4),
        'mejores_params': grid.best_params_
    })

df_resultados = pd.DataFrame(resultados).sort_values('accuracy_cv', ascending=False).reset_index(drop=True)
df_resultados

## Mejor modelo

In [ ]:
mejor_nombre = df_resultados.loc[0, 'modelo']
mejor_modelo = mejores_modelos[mejor_nombre]
print("Mejor modelo:", mejor_nombre)
print(classification_report(y_test, mejor_modelo.predict(X_test)))